# Задание 3. Аномалии и отказ оборудования

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest

from common.utils import pipeline
from common.ETL import save_mart

### Общие ETL шаги

In [2]:
df_sensors = pipeline(
    table_name="pump_sensors",
    date_column="timestamp",
    transform_outliers=False,
)

df_failures = pipeline(
    table_name="pump_failures",
    date_column="failure_date",
    transform_outliers=False,
)

### Поиск аномалий через z-score

In [3]:
sensor_columns = ["vibration", "temperature", "current", "rpm"]

for column in sensor_columns:
    mean_value = df_sensors[column].mean()
    std_value = df_sensors[column].std()
    df_sensors[f"{column}_zscore"] = (df_sensors[column] - mean_value) / std_value

zscore_columns = [f"{column}_zscore" for column in sensor_columns]
df_sensors["anomaly_score"] = df_sensors[zscore_columns].abs().max(axis=1)
df_sensors["is_anomaly"] = df_sensors["anomaly_score"] >= 2

df_sensors[["pump_id", "timestamp", "anomaly_score", "is_anomaly"]].head()

,pump_id,timestamp,anomaly_score,is_anomaly
0,1,2025-10-01 00:00:00,0.726936,False
1,1,2025-10-01 03:00:00,0.750459,False
2,1,2025-10-01 06:00:00,0.703412,False
3,1,2025-10-01 09:00:00,0.726936,False
4,1,2025-10-01 12:00:00,0.679889,False


### Поиск аномалий через Isolation Forest

In [4]:
isolation_forest = IsolationForest(
    contamination=0.1,
    random_state=42,
)

df_sensors["isolation_prediction"] = isolation_forest.fit_predict(df_sensors[sensor_columns])
df_sensors["isolation_score"] = -isolation_forest.score_samples(df_sensors[sensor_columns])
df_sensors["is_isolation_anomaly"] = df_sensors["isolation_prediction"] == -1

mart_pump_anomalies = df_sensors[
    ["pump_id", "timestamp", "date"] + sensor_columns + zscore_columns + [
        "anomaly_score",
        "is_anomaly",
        "isolation_score",
        "is_isolation_anomaly",
    ]
].copy()

mart_pump_anomalies.head()

,pump_id,timestamp,date,vibration,temperature,current,rpm,vibration_zscore,temperature_zscore,current_zscore,rpm_zscore,anomaly_score,is_anomaly,isolation_score,is_isolation_anomaly
0,1,2025-10-01 00:00:00,2025-10-01,2.1,72.3,58.2,1470.0,-0.726936,-0.627952,-0.204416,-0.205792,0.726936,False,0.425897,False
1,1,2025-10-01 03:00:00,2025-10-01,2.0,72.6,58.4,1472.0,-0.750459,-0.568540,-0.140425,-0.133160,0.750459,False,0.423803,False
2,1,2025-10-01 06:00:00,2025-10-01,2.2,73.1,58.6,1474.0,-0.703412,-0.469520,-0.076434,-0.060527,0.703412,False,0.405703,False
3,1,2025-10-01 09:00:00,2025-10-01,2.1,72.8,58.3,1471.0,-0.726936,-0.528932,-0.172421,-0.169476,0.726936,False,0.414362,False
4,1,2025-10-01 12:00:00,2025-10-01,2.3,73.0,58.5,1473.0,-0.679889,-0.489324,-0.108429,-0.096843,0.679889,False,0.404568,False


### Признаки перед отказом

In [5]:
sensors_with_failures = df_sensors.merge(
    df_failures[["pump_id", "failure_date", "failure_type"]],
    on="pump_id",
    how="left"
)

sensors_with_failures["hours_before_failure"] = (
    sensors_with_failures["failure_date"] - sensors_with_failures["timestamp"]
).dt.total_seconds() / 3600

mart_vibration_before_failure = sensors_with_failures[
    sensors_with_failures["hours_before_failure"].between(0, 24)
][[
    "pump_id",
    "timestamp",
    "failure_date",
    "failure_type",
    "hours_before_failure",
    "vibration",
    "temperature",
    "current",
    "rpm",
    "anomaly_score",
    "is_anomaly",
]].copy()

mart_vibration_before_failure.head()

,pump_id,timestamp,failure_date,failure_type,hours_before_failure,vibration,temperature,current,rpm,anomaly_score,is_anomaly
49,1,2025-10-03 03:00:00,2025-10-04 02:00:00,Overheating,23.0,4.5,78.2,61.5,1495.0,0.851438,False
50,1,2025-10-03 06:00:00,2025-10-04 02:00:00,Overheating,20.0,5.0,79.1,62.0,1498.0,1.011416,False
51,1,2025-10-03 09:00:00,2025-10-04 02:00:00,Overheating,17.0,5.6,80.5,62.5,1500.0,1.171394,False
52,1,2025-10-03 12:00:00,2025-10-04 02:00:00,Overheating,14.0,6.3,81.7,63.0,1502.0,1.331371,False
53,1,2025-10-03 15:00:00,2025-10-04 02:00:00,Overheating,11.0,7.2,82.8,63.5,1503.0,1.491349,False


### Risk score по насосам

In [6]:
mart_pump_risk_score = (
    df_sensors
    .groupby("pump_id", as_index=False)
    .agg(
        max_anomaly_score=("anomaly_score", "max"),
        anomaly_count=("is_anomaly", "sum"),
        max_vibration=("vibration", "max"),
        max_temperature=("temperature", "max"),
        max_current=("current", "max"),
        max_rpm=("rpm", "max"),
    )
)

mart_pump_risk_score["failure_probability"] = (
    mart_pump_risk_score["max_anomaly_score"] / mart_pump_risk_score["max_anomaly_score"].max()
).clip(0, 1)

mart_pump_risk_score = mart_pump_risk_score.merge(
    df_failures[["pump_id", "failure_date", "failure_type"]],
    on="pump_id",
    how="left"
)

mart_pump_risk_score.sort_values("failure_probability", ascending=False)

,pump_id,max_anomaly_score,anomaly_count,max_vibration,max_temperature,max_current,max_rpm,failure_probability,failure_date,failure_type
2,5,3.601353,5,20.5,89.3,65.5,1534.0,1.000000,2025-10-04 08:00:00,Electrical fault
0,1,2.035274,1,9.1,85.1,65.2,1508.0,0.565142,2025-10-04 02:00:00,Overheating
1,3,1.658442,0,9.4,81.4,60.0,1480.0,0.460505,2025-10-04 06:00:00,Bearing vibration


### Сохранение витрин в БД

In [7]:
save_mart(mart_pump_anomalies, "mart_pump_anomalies")
save_mart(mart_vibration_before_failure, "mart_vibration_before_failure")
save_mart(mart_pump_risk_score, "mart_pump_risk_score")